# Flow Matching on $\mathrm{SE}(2)$
In this notebook, we perform flow matching on the roto-translation group $\mathrm{SE}(2)$, corresponding to Figure 1 in ["Flow Matching on Lie Groups"](https://arxiv.org/abs/2504.00494).

In [ ]:
import torch
import numpy as np
from lieflow.groups import SE2
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance
)
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"

In [ ]:
se2 = SE2()

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_normals(N):
    translations = torch.randn(N, 2, device=device)
    rotations = 2 * torch.pi * torch.rand(N, device=device)
    g = torch.hstack((translations, rotations[..., None]))
    return g

def generate_uniforms_on_circle(N, centre=(0., 0.), ε=0.05):
    centre = torch.tensor(centre, device=device)
    angles = (torch.rand(N, device=device) * 2 * torch.pi)[..., None]
    gs = torch.hstack((torch.cos(angles) + centre[0], torch.sin(angles) + centre[1], angles))
    return gs + ε * torch.randn(gs.shape, device=device)

def generate_line(N, d=0., w=1., horizontal=True, ε=0.05):
    if horizontal:
        xs = 2 * w * (torch.rand(N, device=device) - 0.5)
        ys = torch.ones(N, device=device) * d
        angles = torch.ones(N, device=device) * np.pi / 2.
    else:
        xs = torch.ones(N, device=device) * d
        ys = 2 * w * (torch.rand(N, device=device) - 0.5)
        angles = torch.zeros(N, device=device)
    gs = (torch.hstack((xs[..., None], ys[..., None], angles[..., None])))
    return gs + ε * torch.randn(gs.shape, device=device)

def plot_g(ax, g, **kwargs):
    g = g.to("cpu")
    return ax.quiver(g[..., 0], g[..., 1], torch.cos(g[..., 2]), torch.sin(g[..., 2]), width=0.01, **kwargs)

In [ ]:
tests = ("horizontal_line_to_vertical_line", "vertical_line_to_circle")

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "horizontal_line_to_vertical_line":
            generate_g_0 = partial(generate_line, d=-2.5, w=1.5, horizontal=True, ε=ε)
            generate_g_1 = partial(generate_line, d=2.5, w=1.5, horizontal=False, ε=ε)
        case "vertical_line_to_vertical_line":
            generate_g_0 = partial(generate_line, d=-2.5, w=1.5, horizontal=False, ε=ε)
            generate_g_1 = partial(generate_line, d=2.5, w=1.5, horizontal=False, ε=ε) 
        case "normal_to_circle":
            generate_g_0 = partial(generate_normals, ε=ε)
            generate_g_1 = partial(generate_uniforms_on_circle, ε=ε)
        case "vertical_line_to_circle":
            generate_g_0 = partial(generate_line, d=-2.5, w=2., horizontal=False, ε=ε)
            generate_g_1 = partial(generate_uniforms_on_circle, centre=np.array((1.25, 0.)), ε=ε)
    return generate_g_0, generate_g_1

# Models

## Training

In [ ]:
EPSILON = 0.03
N = 2**14
BATCH_SIZE = 2**10
EPOCHS = len(tests) * [50]
WEIGHT_DECAY = 0.
LEARNING_RATE = 1e-2
H = 64 # Width
L = 3 # Number of layers is L + 2

In [ ]:
def train_model(g_0s, g_1s, epochs, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(g_0s, g_1s), batch_size=batch_size, shuffle=True
    )

    model_FM = get_model_FM(se2, H=H, L=L).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), learning_rate, weight_decay=weight_decay)
    loss = LogarithmicDistance(torch.tensor([1., 1., 1.], device=device))

    losses_FM = np.zeros(epochs)
    for i in tqdm(range(epochs)):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, loss, show_progress=False)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot(losses_FM)
    ax.set_title("Batch Loss Flow Matching")
    ax.set_xscale("log")
    ax.set_yscale("log")

    return model_FM

In [ ]:
models_FM = {}
for i, test in enumerate(tests):
    print(test)
    generate_g_0, generate_g_1 = data_generator(test, ε=EPSILON)

    g_0s = generate_g_0(N)
    g_1s = generate_g_1(N)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    plot_g(ax, g_0s[:32], color="red")
    plot_g(ax, g_1s[:32], color="green")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)

    models_FM[test] = train_model(g_0s, g_1s, epochs=EPOCHS[i], batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)

## Testing

In [ ]:
def create_animations(tests, models_FM, N=2**5):
    g_0s_test = {}
    g_ts_test = {}
    for test in tests:
        generate_g_0, _ = data_generator(test, ε=EPSILON)
        g_0s_test[test] = generate_g_0(N).to(device)
        g_ts_test[test] = g_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    N_steps = 120
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': 'Flow Matching SE(2)', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)

        with writer.saving(fig, f"output/flow_matching_SE2_{test}.mp4", dpi=150):
            quiver = ax.quiver([], [], [], [])
            for frame in tqdm(range(N_steps)):
                quiver.remove()
                quiver = plot_g(ax, g_ts_test[test], color="tab:blue")
                ax.set_title(fr"$\mathfrak{{X}}_{{{t:.2f}}}$")
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    g_ts_test[test] = models_FM[test].step(g_ts_test[test], torch.tensor([t], device=device), Δt)

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N=2**5)

In [ ]:
t = 0
N_steps = 240
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

g_0s_test = {}
g_1s_test = {}
g_ts_test = {}
for test in tests:
    generate_g_0, generate_g_1 = data_generator(test, ε=EPSILON)
    g_0s_test[test] = generate_g_0(N_samples).to(device)
    g_1s_test[test] = generate_g_1(N_samples).to(device)
    g_ts_test[test] = g_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j])
        a.set_xlim(-3, 3)
        a.set_ylim(-3, 3)
        a.set_xticks([])
        a.set_yticks([])
        a.set_aspect("equal")
        ax[i].append(a)
        
ax[0][0].set_title(r"$\mathfrak{X}_0$")
ax[0][1].set_title(r"$\mathfrak{X}_t$")
ax[0][2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_g(ax[i][0], g_0s_test[test][:N_samples], color="tab:blue")
    plot_g(ax[i][2], g_1s_test[test][:N_samples], color="tab:blue")

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            g_ts_test[test] = models_FM[test].step(g_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:
            plot_g(ax[i][1], g_ts_test[test][:N_samples_shown], color=colors[k], alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_SE2.pdf", bbox_inches="tight")